<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Operating-Systems/11-virtualization-containers-and-modern-kernels.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Operating Systems guideline](Operating-Systems.html)


## **Virtualization, Containers, and Modern Kernel Design**

The previous chapter built protection domains inside one operating system: credentials identify subjects, permissions and capabilities constrain authority, namespaces alter views, cgroups govern resources, and seccomp or LSM policy narrows operations. This chapter asks what happens when the boundary moves **below an entire operating system**, or when those same-kernel mechanisms are composed into a deployable container.

Three words that are often treated as synonyms describe different abstractions:

- **machine virtualization** presents one or more virtual machines, each with virtual CPUs, guest-physical memory and devices on which a guest kernel can run;
- **operating-system-level virtualization** gives processes isolated views and resource domains while they share one host kernel;
- **emulation** reproduces an interface or instruction set in software and may therefore run code built for different hardware, usually with more translation work.

The difference is not merely packaging. It determines which kernel handles a syscall, which scheduler ultimately assigns a physical CPU, whose page tables describe an address, which software interprets device commands, how failure propagates, and what belongs to the Trusted Computing Base (TCB).

![Native processes, containers, and virtual machines place their principal isolation boundaries at different layers.](assets/virtualization-boundaries.svg){fig-alt="Three layered stacks compare native process isolation, containers sharing a host kernel, and virtual machines with separate guest kernels above a hypervisor." width="98%"}

*Figure: original explanatory diagram based on the [Linux KVM API](https://docs.kernel.org/virt/kvm/api.html), the [OCI Linux runtime configuration](https://specs.opencontainers.org/runtime-spec/config-linux/), and [NIST SP 800-190](https://csrc.nist.gov/pubs/sp/800/190/final).*

The running case remains:

```bash
cat input.txt | grep kernel > result.txt
```

Natively, both programs issue syscalls to the host kernel. In a Linux container, they still issue those same syscalls to the host kernel, but the kernel resolves names, identities and resource accounting through container-specific namespace and cgroup state. In a VM, they issue syscalls to a **guest** kernel; that kernel's privileged instructions, memory mappings and I/O eventually cross a hypervisor-managed virtual-hardware boundary.

A useful question for the entire chapter is therefore:

> Which layer owns the abstraction, which layer enforces the boundary, and which layers must remain correct for the claimed isolation to hold?

### **Why Virtualize a Computer?**

To **virtualize** a resource is to insert a control layer that presents each consumer with a useful logical version while multiplexing, translating or constraining access to the physical version. Virtual memory virtualizes an address space for a process. A hypervisor extends this idea to a machine interface: CPU privilege, physical memory, interrupts, timers and devices become per-VM state.

Virtualization is useful because software often wants contradictory properties. A workload wants to believe that it owns a stable machine, while an operator wants to consolidate many workloads, move them between hosts, meter them independently, recover them from snapshots and limit faults. The virtual-machine monitor (VMM) reconciles those views.

Virtualization should not be sold as automatic security. A guest is isolated only against the adversary named in the threat model. Device emulators, hypercalls, shared-memory channels, management APIs, firmware, host kernel code and hardware all expose interfaces. Moving a boundary can remove a guest kernel from another tenant's TCB, while simultaneously adding a VMM and management plane to the infrastructure TCB.

#### **Consolidation, Isolation, Portability, and Management**

**Consolidation** allows workloads with different peak times to share hardware. If each service reserves a physical host for its peak, average utilization may be low. A hypervisor can schedule several vCPUs on fewer physical CPUs, allocate guest memory dynamically and share storage or network devices. Overcommit is a policy, not free capacity: when all guests demand their advertised resources simultaneously, the system must reclaim, throttle, swap, balloon, queue, or fail.

**Isolation** creates separate fault and administration domains. One VM can run a different kernel version or even a different operating system. A crash in a guest kernel normally remains inside that VM because the guest kernel does not own host page tables or physical devices directly. The qualifier matters: a hypervisor escape, unsafe device assignment, firmware defect or vulnerable management service can still cross the boundary.

**Portability** packages machine state behind a stable virtual platform. A VM image can include a bootloader, kernel, userspace and filesystem. A snapshot can capture memory and device state. Live migration can transfer a running VM between compatible hosts. Portability still has contracts: CPU features, virtual machine type, firmware, endianness, page size, device model, storage availability and network identity must remain compatible.

**Management** makes machine lifecycle programmable. Operators can create, pause, inspect, snapshot, clone, meter, migrate and destroy logical machines through an API. This supports testing, disaster recovery, elastic fleets and reproducible environments. It also creates a highly privileged control plane; compromise of the API or image supply chain can be more damaging than compromise of one guest.

| Goal | Mechanism that helps | Hidden cost or failure mode |
|---|---|---|
| Consolidation | vCPU scheduling, memory overcommit, shared backends | Correlated demand, noisy neighbours, host-wide failure |
| Isolation | Hardware privilege, nested paging, IOMMU, narrow VMM | Escape vulnerabilities, side channels, large device models |
| Portability | Stable virtual hardware, images, snapshots | Compatibility matrices, stale secrets, device state |
| Recovery | Checkpoint, replication, migration | Crash-consistent storage and external dependencies |
| Experimentation | Disposable VMs and reversible snapshots | Snapshot trees, hidden state and non-reproducible manual changes |
| Fleet management | Declarative machine configuration and APIs | Privileged management plane becomes a high-value TCB component |

The overhead must be decomposed rather than summarized as "virtualization is slow." Compute-heavy guest user code can execute directly on hardware. Costs appear when execution exits to the VMM, translations miss in the TLB, virtual I/O copies or notifies excessively, host and guest schedulers conflict, or resources are overcommitted. A warm, well-configured VM can be close to native for one workload while showing severe tail latency for another.

<details>
<summary><strong>Inspect whether a Linux host exposes hardware virtualization</strong></summary>

```bash
# Identify the environment first. This may report none, kvm, vmware,
# microsoft, wsl, docker, podman, and other recognized environments.
systemd-detect-virt || true

# CPU flags commonly include vmx on Intel or svm on AMD when the
# processor and firmware expose hardware virtualization support.
lscpu | grep -E 'Virtualization|Hypervisor vendor|Flags'

# KVM userspace opens /dev/kvm. Its permissions determine who may
# create VMs; presence alone does not grant the current user access.
ls -l /dev/kvm 2>/dev/null || echo '/dev/kvm is not available'

# Loaded modules show the generic KVM core and an architecture backend.
lsmod | grep -E '^kvm(_intel|_amd)?\b' || true

# Do not interpret a missing flag inside a VM as proof that the physical
# CPU lacks support: the outer hypervisor may hide nested virtualization.
```

</details>

This inspection separates four states that are often confused: the physical CPU implements an extension; firmware enables it; the host kernel exposes a virtualization API; and policy permits this process to use that API.

### **Hypervisor Architectures**

A **hypervisor** or **VMM** is the privileged control layer that creates virtual CPUs, establishes guest-memory ownership, delivers virtual interrupts, mediates devices and switches execution between guests and host components. It may be a compact standalone system, a privileged kernel subsystem plus user-space device model, or a hosted application using a general-purpose OS.

The architecture is easier to understand by following responsibility:

1. who schedules vCPUs onto physical CPUs;
2. who owns the second-stage page tables;
3. who emulates or services devices;
4. who performs host resource allocation;
5. where management code and drivers execute; and
6. which component can read or modify guest state.

#### **Type 1 and Type 2 Hypervisors**

A **Type 1** hypervisor runs as the primary privileged control layer on the hardware. Guest operating systems execute above it, sometimes alongside a privileged management or driver domain. This structure can keep ordinary host applications outside the lowest control layer, but a large management domain or driver stack may still belong to the practical TCB.

A **Type 2** hypervisor runs as an application on a host operating system. It relies on the host for files, devices, memory allocation, scheduling and user interaction. This is convenient for desktop development because normal host drivers and services remain available, but guest execution competes through both host and guest scheduling layers.

![Type 1 and Type 2 describe placement, while systems such as KVM distribute VMM responsibilities across kernel and user space.](assets/hypervisor-architectures.svg){fig-alt="Layered comparison of bare-metal Type 1 and hosted Type 2 hypervisors, followed by a note that KVM and QEMU form a hybrid architecture." width="97%"}

*Figure: original explanatory diagram based on the [KVM API](https://docs.kernel.org/virt/kvm/api.html), [QEMU system documentation](https://www.qemu.org/docs/master/system/index.html), and the Xen [architecture introduction](https://xenbits.xen.org/docs/latest/admin-guide/introduction.html).*

The taxonomy is useful but imperfect. With **KVM**, Linux provides the `/dev/kvm` interface, vCPU execution, memory-virtualization controls and interrupt support. A user-space VMM such as QEMU creates the VM, maps guest RAM, supplies a virtual machine model and implements many device paths. Linux is simultaneously a host kernel, resource manager, driver platform and part of the hypervisor solution. Calling it only Type 1 or Type 2 hides more than it explains.

The security consequence is similarly architectural. In a hosted VMM, host kernel drivers can affect every VM. In a split design, a user-space device model may be sandboxed separately, reducing the consequence of a parser defect. In a design with a privileged driver domain, that domain's authority and communication protocol must be included in the threat model.

#### **Full Virtualization and Paravirtualization**

**Full virtualization** presents an interface on which an unmodified guest operating system can run. The VMM preserves the architectural behavior the guest expects: privileged state, interrupts, timers, page tables and devices. "Full" does not mean every physical detail is reproduced. The virtual machine may expose a deliberately selected CPU model and synthetic devices, as long as the guest-visible contract is valid.

**Paravirtualization** changes the guest-to-hypervisor interface so the guest cooperates. Instead of attempting an operation that must be trapped and emulated, a modified guest can issue a **hypercall**. A paravirtualized driver such as virtio uses shared queues designed for virtual I/O rather than pretending to be a legacy physical controller.

Modern systems are hybrids:

- ordinary guest user instructions execute directly;
- hardware virtualization controls privileged guest execution;
- nested paging translates guest memory without maintaining every mapping in software;
- paravirtualized clock, interrupt and I/O interfaces avoid expensive emulation paths; and
- selected devices may be assigned directly through an IOMMU.

| Interface | Guest modification | Main advantage | Main cost |
|---|---:|---|---|
| Fully emulated legacy hardware | None | Maximum compatibility, including old guests | Frequent exits and complex device emulation |
| Hypercall-based paravirtualization | Kernel awareness required | Explicit, efficient privileged interface | Guest dependency on hypervisor ABI |
| Paravirtualized device | Driver required | Batched shared-memory data path | Backend and guest driver must agree on protocol |
| Hardware-assisted full virtualization | Usually none in core kernel | Direct execution with controlled exits | Exit and nested-translation costs remain |
| Device passthrough | Native device driver | Near-native data path | Harder isolation, reset, sharing and migration |

The comparison should distinguish **semantic compatibility** from **implementation path**. An unmodified application can run in a guest whose kernel contains virtio drivers. From the application's perspective the machine remains compatible, while the kernel and VMM cooperate for performance.

### **CPU Virtualization**

CPU virtualization gives each VM one or more **virtual CPUs (vCPUs)** with architectural registers, privilege state, interrupt state, control registers and timers. A vCPU is a schedulable software object. When it is running, a physical CPU executes guest instructions in a hardware-constrained guest mode; when it is not, its state is saved and another host task or vCPU can run.

This introduces two schedulers. The guest scheduler chooses which guest thread runs on a vCPU. The host scheduler chooses when that vCPU thread runs on a physical CPU. A guest may believe that two vCPUs are executing simultaneously even when the host deschedules one, which can distort spin locks, timekeeping and latency. CPU pinning, topology exposure and paravirtualized yield or clock mechanisms help, but they exchange flexibility for predictability.

#### **Trap-and-Emulate and Hardware Assistance**

The classical virtualization requirement is that the VMM must regain control whenever a guest attempts an operation that can affect the machine outside its virtual state. In an ideal **trap-and-emulate** path:

1. ordinary, non-sensitive guest instructions execute directly;
2. a sensitive privileged operation traps before changing forbidden host state;
3. the VMM reads the exit reason and guest operands;
4. it updates virtual state, injects an exception, performs an allowed host action, or blocks the vCPU; and
5. it resumes the guest at the correct architectural point.

Popek and Goldberg formalized conditions under which a classical instruction set can be efficiently virtualized. Early x86 contained sensitive behaviors that did not always trap cleanly at lower privilege, so VMMs used techniques such as dynamic binary translation. Modern Intel VT-x and AMD-V add explicit guest and hypervisor execution modes, control structures and intercept configuration.

![Hardware assistance lets ordinary guest instructions run directly and transfers control to the VMM only for configured exits.](assets/vm-entry-exit-animated.svg){fig-alt="Animated VM-entry and VM-exit cycle between guest non-root execution and hypervisor root handling of intercepted events." width="96%"}

*Figure: original explanatory diagram based on the [Intel 64 and IA-32 Software Developer's Manual](https://www.intel.com/content/www/us/en/developer/articles/technical/intel-sdm.html), the AMD64 Architecture Programmer's Manual, and the [KVM API](https://docs.kernel.org/virt/kvm/api.html).*

On Intel, a VM control structure records guest state, host state, entry controls, exit controls and the exit reason; AMD provides analogous mechanisms. The terminology "root" and "non-root" is separate from ordinary OS rings. A guest kernel can execute at its expected privilege level **inside non-root mode**, while the hypervisor retains an additional control dimension.

Common causes of VM exits include configured control-register accesses, certain model-specific registers, I/O port accesses, external interrupts, halt instructions, second-stage translation faults and explicit hypercalls. Not every privileged instruction must exit; hardware can maintain guest-owned copies or virtualized behavior directly.

The cost of one exit is not merely saving a few registers. It can include pipeline disruption, VMM dispatch, host scheduling, device-model work, cache and TLB effects, and re-entry. Good virtualization paths therefore:

- avoid unnecessary intercepts;
- batch I/O descriptors and completions;
- use posted or virtualized interrupts where available;
- keep frequently used guest state in hardware structures;
- expose stable paravirtual interfaces for operations that otherwise exit often; and
- account separately for average throughput and tail latency.

<details>
<summary><strong>Minimal x86 KVM control loop in C</strong></summary>

```c
// A compact teaching VMM for Linux/x86. It creates one vCPU, maps one
// page of guest RAM, runs a few real-mode instructions, handles port I/O,
// and stops on HLT. Production VMMs need robust device, interrupt, memory,
// migration, firmware, concurrency, and security models.

#define _GNU_SOURCE
#include <errno.h>
#include <fcntl.h>
#include <linux/kvm.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <sys/ioctl.h>
#include <sys/mman.h>
#include <unistd.h>

static void fail(const char *what) {
    perror(what);
    exit(EXIT_FAILURE);
}

int main(void) {
    // mov dx, 0x3f8; mov al, 'H'; out dx, al; hlt
    const uint8_t guest_code[] = {0xba, 0xf8, 0x03, 0xb0, 0x48, 0xee, 0xf4};

    int kvm = open("/dev/kvm", O_RDWR | O_CLOEXEC);
    if (kvm < 0) fail("open /dev/kvm");
    if (ioctl(kvm, KVM_GET_API_VERSION, 0) != KVM_API_VERSION) {
        fprintf(stderr, "unexpected KVM API version\n");
        return EXIT_FAILURE;
    }

    // A VM file descriptor owns machine-wide state.
    int vm = ioctl(kvm, KVM_CREATE_VM, 0);
    if (vm < 0) fail("KVM_CREATE_VM");

    // Userspace allocates RAM; KVM is told which guest-physical range it backs.
    size_t ram_size = 0x1000;
    uint8_t *ram = mmap(NULL, ram_size, PROT_READ | PROT_WRITE,
                        MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    if (ram == MAP_FAILED) fail("mmap guest RAM");
    memcpy(ram, guest_code, sizeof guest_code);

    struct kvm_userspace_memory_region region = {
        .slot = 0,
        .guest_phys_addr = 0,
        .memory_size = ram_size,
        .userspace_addr = (uintptr_t)ram,
    };
    if (ioctl(vm, KVM_SET_USER_MEMORY_REGION, &region) < 0)
        fail("KVM_SET_USER_MEMORY_REGION");

    // A vCPU file descriptor owns architectural CPU state.
    int vcpu = ioctl(vm, KVM_CREATE_VCPU, 0);
    if (vcpu < 0) fail("KVM_CREATE_VCPU");
    int run_size = ioctl(kvm, KVM_GET_VCPU_MMAP_SIZE, 0);
    if (run_size < (int)sizeof(struct kvm_run)) fail("KVM_GET_VCPU_MMAP_SIZE");
    struct kvm_run *run = mmap(NULL, (size_t)run_size,
                               PROT_READ | PROT_WRITE, MAP_SHARED, vcpu, 0);
    if (run == MAP_FAILED) fail("mmap kvm_run");

    // Start in a simple real-mode layout at guest address zero.
    struct kvm_sregs sregs;
    if (ioctl(vcpu, KVM_GET_SREGS, &sregs) < 0) fail("KVM_GET_SREGS");
    sregs.cs.base = 0;
    sregs.cs.selector = 0;
    if (ioctl(vcpu, KVM_SET_SREGS, &sregs) < 0) fail("KVM_SET_SREGS");

    struct kvm_regs regs = {.rip = 0, .rflags = 0x2};
    if (ioctl(vcpu, KVM_SET_REGS, &regs) < 0) fail("KVM_SET_REGS");

    for (;;) {
        // KVM_RUN enters guest mode. The ioctl returns on a userspace exit.
        if (ioctl(vcpu, KVM_RUN, 0) < 0) {
            if (errno == EINTR) continue;
            fail("KVM_RUN");
        }

        switch (run->exit_reason) {
        case KVM_EXIT_IO:
            // For this demo, accept one-byte output to the serial I/O port.
            if (run->io.direction == KVM_EXIT_IO_OUT &&
                run->io.port == 0x3f8 && run->io.size == 1) {
                uint8_t *data = (uint8_t *)run + run->io.data_offset;
                fwrite(data, 1, run->io.count, stdout);
                fflush(stdout);
            } else {
                fprintf(stderr, "unexpected I/O exit\n");
                return EXIT_FAILURE;
            }
            break;
        case KVM_EXIT_HLT:
            putchar('\n');
            return EXIT_SUCCESS;
        case KVM_EXIT_FAIL_ENTRY:
            fprintf(stderr, "failed VM entry: 0x%llx\n",
                    (unsigned long long)run->fail_entry.hardware_entry_failure_reason);
            return EXIT_FAILURE;
        case KVM_EXIT_INTERNAL_ERROR:
            fprintf(stderr, "KVM internal error\n");
            return EXIT_FAILURE;
        default:
            fprintf(stderr, "unhandled KVM exit reason: %u\n", run->exit_reason);
            return EXIT_FAILURE;
        }
    }
}
```

Compile and run on a Linux x86 host with KVM access:

```bash
cc -std=c11 -Wall -Wextra -O2 tiny_kvm.c -o tiny_kvm
./tiny_kvm
# Expected output: H
```

</details>

The example makes the control split concrete. KVM executes guest CPU state and reports structured exits through `struct kvm_run`; userspace decides what virtual machine those exits mean. The code is intentionally too small to boot an OS and should not be treated as a secure VMM.

### **Memory Virtualization**

A guest operating system believes that it maps **guest virtual addresses (GVA)** to **guest physical addresses (GPA)**. The host must then map those guest-physical frames to **host physical addresses (HPA)**. The actual translation is therefore a composition:

$$
\operatorname{HPA}
= T_{\text{nested}}\!\left(T_{\text{guest}}(\operatorname{GVA})\right).
$$

The guest owns the first mapping as part of its process abstraction. The hypervisor owns the second mapping because it must prevent one VM from addressing another VM's or the host's pages. Effective permissions are the intersection of both stages: a writable guest PTE cannot override a read-only nested mapping.

Virtualized memory also includes allocation and lifecycle policy. Guest RAM may be backed by anonymous host memory, huge pages, files, shared memory or device memory. The host can overcommit it, deduplicate it under carefully considered security policy, reclaim it through ballooning, pin it for DMA, or track dirty pages for migration. "The VM has 8 GB" is a guest-visible promise whose physical backing can change over time.

#### **Shadow Page Tables and Nested Translation**

With **shadow page tables**, the hypervisor builds hardware-visible mappings that directly translate GVA to HPA. It validates the guest's intended GVA-to-GPA mapping, combines it with host ownership and writes a shadow entry. The steady-state hardware lookup is simple, but the VMM must notice guest page-table changes, maintain consistency and invalidate stale shadows.

With **nested paging**, called Extended Page Tables (EPT) by Intel and Nested Page Tables (NPT) by AMD, hardware performs two translation stages. The guest can update its own page tables normally. The hypervisor configures the second-stage root and handles faults when a GPA is absent or violates nested permissions.

![Shadow page tables compose mappings in software, while nested paging lets the MMU walk guest and hypervisor-owned tables.](assets/nested-address-translation.svg){fig-alt="Comparison of shadow GVA-to-HPA mappings and nested GVA-to-GPA-to-HPA translation with TLB and page-walk-cache notes." width="98%"}

*Figure: original explanatory diagram based on the [Intel SDM EPT description](https://www.intel.com/content/www/us/en/developer/articles/technical/intel-sdm.html), AMD's NPT architecture, and the Linux [KVM memory-region API](https://docs.kernel.org/virt/kvm/api.html).*

A cold two-dimensional walk can be expensive. For four-level guest tables and four-level nested tables, reading each guest page-table entry requires translating the GPA of the guest page-table page. In a simplified worst case, this can involve up to 24 translation-structure memory references before the final data access. This is **not** paid on every load or store: TLBs, nested TLB entries, page-walk caches and large pages exist precisely to keep the common path short.

The important performance events are therefore:

- guest TLB miss and nested walk depth;
- host decisions that invalidate nested mappings;
- vCPU migration between physical CPUs and associated translation context;
- guest and host page sizes, including fragmentation tradeoffs;
- NUMA placement of vCPU threads and backing memory;
- dirty-bit collection for migration; and
- memory overcommit, ballooning, host swap and guest swap interacting at once.

Double paging can create pathologies. If the guest swaps a page into its virtual disk while the host swaps the guest-memory page that contains the relevant cache or buffer, both layers perform replacement without seeing the other's full objective. Production policy usually avoids uncontrolled host swapping for latency-sensitive VMs and coordinates limits, ballooning and NUMA placement.

<details>
<summary><strong>Simulate two-stage translation and permission intersection in Python</strong></summary>

```python
from dataclasses import dataclass


@dataclass(frozen=True)
class Mapping:
    frame: int
    read: bool = True
    write: bool = False
    execute: bool = False


PAGE_SIZE = 4096


def translate(gva: int, access: str,
              guest_pt: dict[int, Mapping],
              nested_pt: dict[int, Mapping]) -> int:
    """Translate GVA -> GPA -> HPA and enforce both permission stages."""
    guest_vpn, offset = divmod(gva, PAGE_SIZE)

    # Stage 1 is selected by the guest OS for one of its processes.
    guest = guest_pt.get(guest_vpn)
    if guest is None:
        raise MemoryError("guest page fault: no GVA -> GPA mapping")

    # The guest frame number is a guest-physical page number.
    guest_pfn = guest.frame

    # Stage 2 is controlled by the hypervisor and enforces VM ownership.
    nested = nested_pt.get(guest_pfn)
    if nested is None:
        raise MemoryError("nested fault: GPA has no host backing")

    # Access is allowed only if BOTH stages grant it.
    permission = {
        "read": guest.read and nested.read,
        "write": guest.write and nested.write,
        "execute": guest.execute and nested.execute,
    }
    if not permission[access]:
        raise PermissionError(f"{access} denied by composed permissions")

    return nested.frame * PAGE_SIZE + offset


guest_pt = {
    0x4: Mapping(frame=0x20, read=True, write=True),
}
nested_pt = {
    # Hypervisor deliberately exposes this page as read-only.
    0x20: Mapping(frame=0x9A, read=True, write=False),
}

gva = 0x4 * PAGE_SIZE + 123
print(hex(translate(gva, "read", guest_pt, nested_pt)))  # 0x9a07b

try:
    translate(gva, "write", guest_pt, nested_pt)
except PermissionError as error:
    print(error)  # guest write permission cannot override nested read-only
```

</details>

The simulator omits levels, accessed/dirty bits, huge pages, caches and invalidation, but it preserves the security invariant: the guest controls its virtual abstraction only within host-owned guest-physical memory.

### **I/O Virtualization**

CPU and memory virtualization are not enough to boot a useful guest. A kernel expects timers, interrupt controllers, storage, network interfaces, consoles, entropy sources and often accelerators. A VMM must expose those devices while preserving isolation and acceptable throughput.

I/O is difficult because devices are concurrent state machines. They perform DMA without the CPU copying each byte, raise interrupts asynchronously, retain internal queues and firmware state, and may not support reset or migration cleanly. A virtual device must preserve ordering, completion, error and reset semantics across guest, VMM, host driver and physical controller.

#### **Emulation, Paravirtualized Devices, and Device Passthrough**

**Device emulation** presents a device whose register and descriptor contract an existing guest driver already understands. Guest MMIO or port I/O exits to a device model, which validates the request and performs host operations. This is invaluable for firmware and old guests, but faithfully reproducing legacy devices creates code and attack surface in a high-trust component.

**Paravirtualized I/O** exposes a device designed for virtualization. Virtio, standardized by OASIS, uses shared-memory queues. The guest writes descriptors that identify buffers, publishes descriptor heads in an available ring and optionally notifies the backend. The backend validates and processes them, then publishes completions in a used ring. Batching and notification suppression reduce exits.

**Device passthrough** assigns a physical function or an SR-IOV virtual function to a guest. An IOMMU translates and restricts device DMA so the assigned device cannot address arbitrary host memory; interrupt remapping constrains interrupt delivery. The data path can approach native performance, but device reset, sharing, observability, dirty-page tracking and live migration become harder.

![Emulation, virtio-style paravirtualization, and passthrough occupy different points between compatibility and direct hardware access.](assets/virtual-io-strategies.svg){fig-alt="Three virtual I/O stacks compare legacy device emulation, paravirtualized virtqueue transport, and IOMMU-protected device passthrough." width="98%"}

*Figure: original explanatory diagram based on the [Virtio 1.2 specification](https://docs.oasis-open.org/virtio/virtio/v1.2/virtio-v1.2.html), Linux [VFIO documentation](https://docs.kernel.org/driver-api/vfio.html), and QEMU device-model documentation.*

![A split virtqueue moves buffer descriptors and indices rather than copying every payload through a register interface.](assets/virtqueue-animated.svg){fig-alt="Animated virtio request lifecycle from guest driver through descriptor and available rings to backend processing, used ring, and guest completion." width="96%"}

*Figure: original explanatory diagram based on the OASIS [Virtio split virtqueue format](https://docs.oasis-open.org/virtio/virtio/v1.2/virtio-v1.2.html#x1-430005) and Linux [virtio implementation documentation](https://docs.kernel.org/driver-api/virtio/virtio.html).*

The ring is a concurrent protocol. The guest must make descriptor contents visible before publishing the available index. The backend must make completion data visible before advancing the used index. Both sides need bounds checks because queue indices, lengths, flags and guest addresses cross a trust boundary. Shared memory improves performance; it does not make inputs trustworthy.

| Strategy | Data/control path | Compatibility | Performance potential | Isolation and operations |
|---|---|---|---|---|
| Emulation | Exit to software device model | Highest | Lowest for chatty legacy interfaces | Easy snapshot model; large parser/state surface |
| Virtio / paravirtual | Shared queues plus notifications | Needs guest driver | High with batching and vhost | Mediated protocol; migratable state must be versioned |
| Passthrough | Guest driver to assigned function | Device-specific native driver | Near native | IOMMU critical; reset, sharing and migration are difficult |
| SR-IOV virtual function | Hardware-created per-guest function | Vendor driver | High | Better sharing, but hardware/firmware and migration constraints remain |

<details>
<summary><strong>Inspect virtual and assigned devices on a Linux guest or host</strong></summary>

```bash
# PCI identity, selected kernel driver, and alternative modules.
lspci -nnk

# Virtio devices commonly bind to drivers with virtio in the name.
find /sys/bus/virtio/devices -maxdepth 2 -type l -name driver \
  -exec sh -c 'printf "%s -> " "$1"; readlink "$1"' sh {} \; 2>/dev/null

# IOMMU groups are the unit exposed to VFIO. Devices in one group may
# not be independently isolated by the platform topology.
for group in /sys/kernel/iommu_groups/*; do
  [ -d "$group/devices" ] || continue
  printf 'group %s: ' "${group##*/}"
  find "$group/devices" -mindepth 1 -maxdepth 1 -printf '%f '
  printf '\n'
done

# A driver binding does not prove safe assignment. Verify IOMMU enablement,
# group composition, reset support, interrupt remapping, firmware, and policy.
```

</details>

### **Virtual-Machine Lifecycle and Migration**

A VM is not only a running CPU. Its lifecycle includes image selection, machine type, firmware, vCPU topology, memory backing, virtual and assigned devices, credentials, network identity, start, pause, snapshot, migration, reset and destruction. Every transition must define which state is quiescent and who owns it.

A **snapshot** captures enough state to resume from a point in time. A disk-only snapshot taken while applications continue writing may be merely crash-consistent. A memory snapshot must coordinate CPU state, virtual devices, timers and storage ordering. Restoring an old snapshot can also restore expired credentials, reused randomness, old counters or a network identity that conflicts with the live world.

**Live migration** moves a running VM while keeping service interruption small. In pre-copy migration, the source first copies RAM while the VM runs, then repeatedly copies pages dirtied during prior rounds. Finally it pauses the VM, transfers the remaining dirty pages plus CPU and device state, starts the destination and cleans up the source only after acknowledgement.

If a round starts with $D_k$ dirty bytes, the guest dirties memory at rate $r$ bytes/s, and usable migration bandwidth is $B$ bytes/s, a simplified next-round estimate is:

$$
D_{k+1} \approx r\frac{D_k}{B}.
$$

The ratio $r/B$ explains convergence. If it is well below one, each round shrinks. If the guest dirties memory as fast as the link transfers it, the dirty set may not converge and the operator must tolerate more downtime, throttle dirtying, compress or deduplicate pages, increase bandwidth, or switch strategy.

In **post-copy**, the destination resumes before all memory arrives. Access to a missing page faults and requests it from the source while background transfer continues. This bounds duplicate transfer and handles non-converging dirty workloads, but page faults increase latency and a failure before all state reaches the destination is more dangerous because authoritative memory is split across hosts.

![Pre-copy minimizes downtime by iterating dirty pages; post-copy resumes early and fetches missing pages on demand.](assets/live-migration-precopy-postcopy.svg){fig-alt="Two live-migration timelines compare iterative pre-copy and fault-driven post-copy with their convergence, downtime and failure tradeoffs." width="98%"}

*Figure: original explanatory diagram based on the official QEMU [migration framework](https://www.qemu.org/docs/master/devel/migration/), [post-copy documentation](https://www.qemu.org/docs/master/devel/migration/postcopy.html), and [VFIO migration state](https://www.qemu.org/docs/master/devel/migration/vfio.html).*

RAM is only the largest obvious state. Correct migration also requires:

- vCPU registers, interrupt and timer state;
- a compatible destination CPU feature model;
- versioned virtual-device state and in-flight requests;
- storage that is shared, replicated or block-migrated consistently;
- network reconnection without duplicate machine identity;
- firmware and machine-type compatibility;
- dirty tracking for DMA-capable devices; and
- an atomic ownership decision so both copies do not run independently.

<details>
<summary><strong>Model whether a pre-copy workload converges</strong></summary>

```python
def precopy_rounds(memory_gib: float, dirty_mib_s: float,
                   bandwidth_mib_s: float, stop_mib: float,
                   max_rounds: int = 12):
    """Return each simplified pre-copy round and estimated final downtime."""
    remaining = memory_gib * 1024
    rounds = []

    for number in range(max_rounds):
        transfer_seconds = remaining / bandwidth_mib_s
        dirtied_during_round = dirty_mib_s * transfer_seconds
        rounds.append({
            "round": number,
            "sent_mib": remaining,
            "seconds": transfer_seconds,
            "redirtied_mib": dirtied_during_round,
        })

        remaining = min(memory_gib * 1024, dirtied_during_round)
        if remaining <= stop_mib:
            break

    # During stop-and-copy the vCPUs are paused while the final set moves.
    downtime_seconds = remaining / bandwidth_mib_s
    return rounds, remaining, downtime_seconds


for dirty_rate in (50, 300, 900):
    rounds, final_dirty, downtime = precopy_rounds(
        memory_gib=8,
        dirty_mib_s=dirty_rate,
        bandwidth_mib_s=1000,
        stop_mib=64,
    )
    print(
        f"dirty={dirty_rate:>3} MiB/s "
        f"rounds={len(rounds):>2} final={final_dirty:7.1f} MiB "
        f"downtime~{downtime * 1000:6.1f} ms"
    )
```

</details>

This model intentionally ignores compression, zero-page detection, parallel channels, network contention, device state and changing dirty rate. Its purpose is to expose the controlling ratio, not predict a production maintenance window.

### **Container Isolation**

A Linux container is a set of ordinary host processes created with a coordinated isolation and resource configuration. It does not normally boot a private kernel and does not need a virtual CPU or emulated interrupt controller. The executable enters the same host kernel through the same syscall ABI as a native process.

The container abstraction is supplied by the **composition** of mechanisms discussed in the previous chapter:

- namespaces select the process's views of PIDs, mounts, users, networks, IPC objects, host names, cgroup paths and sometimes clocks;
- cgroup v2 places processes in a resource-accounting and control hierarchy;
- credentials, user-ID mappings and capabilities define kernel-recognized authority;
- `no_new_privs`, seccomp and LSM policy restrict future privilege and operations;
- mount configuration and a root filesystem define visible files and devices; and
- a runtime converts an image and declarative configuration into a process with those properties.

This chapter does not repeat each mechanism's syscall semantics. The new idea is that **none of them alone is a container**. A mount namespace without cgroup limits changes a view but does not bound memory. A cgroup without namespaces meters a normal host process. A PID namespace does not hide the network. A capability drop does not select a filesystem. The operational boundary is their intersection.

#### **Namespaces, cgroups, Capabilities, and Layered File Systems**

The Open Container Initiative (OCI) runtime specification defines a bundle containing a root filesystem and a `config.json`. On Linux, the configuration can describe the process arguments and environment, namespace types, mounts, user IDs, capabilities, resource limits, seccomp policy and hooks. A low-level runtime such as `runc` or `crun` translates that contract into clone/unshare operations, cgroup placement, mount setup, credential changes and finally `execve()`.

![A container boundary is the intersection of namespaced views, resource governance, reduced authority, policy and filesystem configuration around host processes.](assets/container-isolation-composition.svg){fig-alt="Concentric composition of Linux container mechanisms around a workload, with namespaces, cgroups, capabilities, seccomp, LSM, layered root filesystem and OCI runtime configuration above a shared host kernel." width="97%"}

*Figure: original explanatory diagram based on the [OCI Linux runtime specification](https://specs.opencontainers.org/runtime-spec/config-linux/), Linux [cgroup v2 documentation](https://docs.kernel.org/admin-guide/cgroup-v2.html), and [NIST SP 800-190](https://csrc.nist.gov/pubs/sp/800/190/final).*

The order of setup is security-sensitive. The runtime may need temporary privilege to create namespaces, populate UID/GID mappings, mount a root filesystem and move the process into a cgroup. It must ensure that no less-restricted workload thread runs before restrictions are installed. Descriptor inheritance, writable procfs or sysfs mounts, device nodes and runtime sockets can bypass an otherwise careful namespace layout.

**Layered filesystems** solve a separate packaging problem. Container image layers are immutable filesystem diffs. OverlayFS can combine one or more read-only lower directories with a per-container writable upper directory and present one merged mount:

- lookup uses the upper object when present, otherwise the highest matching lower object;
- reading a lower file can use it directly, so many containers share image data;
- the first write that requires private state performs **copy-up** into the upper layer;
- deletion records a **whiteout** in the upper layer so a lower name remains hidden; and
- a bind-mounted volume can bypass the image's writable layer and expose separate persistent storage.

![OverlayFS reveals immutable lower content until a write copies an object into the container-specific upper layer.](assets/overlayfs-copy-up-animated.svg){fig-alt="Animated OverlayFS copy-up from a read-only lower image layer to a writable upper layer, with merged lookup and whiteout behavior." width="96%"}

*Figure: original explanatory diagram based on the official Linux [OverlayFS documentation](https://docs.kernel.org/filesystems/overlayfs.html), including copy-up, upper/lower lookup and whiteouts.*

An image is therefore not equivalent to a VM disk. It normally contains user-space files and metadata but assumes a compatible host-kernel ABI. It can be content-addressed, cached and shared efficiently, while runtime writes remain separate. Persistence must be explicit: deleting a container usually discards its private upper layer, but does not necessarily delete mounted volumes.

Rootless containers alter the trust relationship by creating a user namespace in which apparent container root maps to an unprivileged host ID. This reduces host authority if the workload or runtime is compromised. It does not make the kernel untrusted or erase kernel attack surface; the host kernel still parses syscalls from the workload.

<details>
<summary><strong>Inspect how a running container is composed</strong></summary>

```bash
# Start a disposable container with an explicit memory and PID limit.
# Podman is shown because rootless operation is common; equivalent Docker
# inspect commands can expose similar state.
podman run --rm -d --name os-boundary \
  --memory 256m --pids-limit 64 \
  --cap-drop all --security-opt no-new-privileges \
  docker.io/library/alpine:latest sleep 600

# Obtain the host PID of the container's initial process.
pid=$(podman inspect --format '{{.State.Pid}}' os-boundary)
printf 'host PID: %s\n' "$pid"

# Namespace links have different inode numbers from the inspecting shell
# when the process belongs to a distinct namespace instance.
printf '%-12s %-24s %-24s\n' TYPE SELF CONTAINER
for ns in cgroup ipc mnt net pid user uts; do
  printf '%-12s %-24s %-24s\n' \
    "$ns" "$(readlink "/proc/self/ns/$ns")" \
    "$(readlink "/proc/$pid/ns/$ns")"
done

# The host sees the process's cgroup path and enforced controllers.
cat "/proc/$pid/cgroup"
cg_rel=$(awk -F: '$1 == "0" {print $3}' "/proc/$pid/cgroup")
cg="/sys/fs/cgroup$cg_rel"
for file in memory.max pids.max cpu.weight; do
  [ -r "$cg/$file" ] && printf '%s=%s\n' "$file" "$(cat "$cg/$file")"
done

# Credentials and syscall restrictions are properties of the host task.
grep -E '^(Uid|Gid|Cap(Inh|Prm|Eff|Bnd|Amb)|NoNewPrivs|Seccomp):' \
  "/proc/$pid/status"

# Mount information reveals overlay and bind-mount composition.
grep -E ' - (overlay|ext4|xfs|btrfs) ' "/proc/$pid/mountinfo" | head

podman stop os-boundary
```

</details>

The commands deliberately inspect host `/proc` and cgroup state rather than trusting a container's self-description. On systems using a VM-backed container engine, the observed PID may belong to a helper or VM instead; that difference is itself evidence that the product boundary is not a plain shared-kernel container.

### **Virtual Machines Versus Containers**

The common comparison "VMs are secure but slow; containers are fast but weak" is too coarse to guide engineering. A VM can have a large device model and permissive management plane. A container can run rootless with a tiny syscall surface and strong LSM policy. A microVM can boot from a snapshot quickly. A container image can carry a huge language runtime. The correct unit of comparison is the concrete boundary and lifecycle.

![Containers, sandboxed containers, microVMs and general-purpose VMs occupy a continuum of state, compatibility and boundary placement.](assets/vm-container-comparison.svg){fig-alt="Design-space plot comparing Linux containers, sandboxed containers, microVMs and general-purpose VMs by lifecycle state and placement of the isolation boundary." width="96%"}

*Figure: original explanatory diagram based on NIST's [container security guide](https://csrc.nist.gov/pubs/sp/800/190/final), the [Firecracker NSDI paper](https://www.usenix.org/conference/nsdi20/presentation/agache), and the Linux [confidential-computing threat model](https://docs.kernel.org/security/snp-tdx-threat-model.html).*

| Dimension | Shared-kernel container | Virtual machine |
|---|---|---|
| Kernel | Host kernel shared by all containers | Guest kernel per VM, controlled by hypervisor |
| Boot path | Create and execute host processes | Initialize virtual hardware, firmware/bootloader and guest kernel, or restore snapshot |
| OS flexibility | Requires compatible host syscall ABI | Can run a different guest OS and kernel policy |
| Image content | Usually userspace plus metadata and layers | Usually bootable disk plus kernel/firmware relationship |
| Isolation boundary | Host process/namespace boundary | Virtual CPU, memory and device boundary below guest kernel |
| Kernel compromise | Can affect sibling containers and host | Guest-kernel compromise normally stays inside VM |
| Resource accounting | Host cgroup and scheduler directly | Host accounts VM/vCPU; guest accounts internal processes |
| Startup and density | Usually lower state and faster creation | Usually more state; microVM and snapshot designs narrow the gap |
| Observability | Host can inspect tasks with namespace-aware tools | Host sees vCPU/VMM behavior; guest sees internal tasks |
| Migration | Recreate process from image/state; app state often external | Full machine checkpoint/migration is a first-class design |
| TCB emphasis | Host kernel, runtime, policy, image and hardware | Hypervisor/VMM, host components, firmware, hardware and management plane |

**Choose a container** when workloads trust the host kernel, need high density and rapid process lifecycle, use a compatible syscall ABI, and can externalize durable state. This is common for services managed as immutable deployments.

**Choose a VM** when tenants should not share a kernel boundary, guest kernels or operating systems differ, machine-level snapshots/migration matter, or legacy software expects a complete boot environment. Stronger boundary placement is especially valuable when untrusted tenants can execute arbitrary kernel-facing code.

**Combine them** when operational packaging and boundary placement solve different problems. A cluster can schedule containers inside tenant VMs. A serverless platform can present a container-like deployment API while placing each workload in a microVM. A sandboxed runtime can intercept syscalls with a user-space kernel before they reach the host. These designs spend extra translation or memory to reduce the host-kernel interface exposed directly to a workload.

Three caveats prevent false confidence:

1. **Isolation strength is not the same as confidentiality from the operator.** A normal hypervisor can inspect guest RAM. Confidential computing changes that threat model.
2. **A smaller image is not automatically a smaller TCB.** Runtime, host kernel, management services and firmware may remain trusted even when the guest filesystem is tiny.
3. **Startup benchmarks are workload-specific.** Cold image fetch, page faults, language initialization, snapshot restore and network readiness can dominate the nominal container or VM creation time.

### **Comparing Kernel Architectures**

Virtualization asks where to place the machine boundary. Kernel architecture asks where to place operating-system services and policy **within or beside that boundary**. These choices affect call paths, fault containment, TCB size, compatibility and the freedom to specialize.

An architecture should be evaluated with concrete questions:

- Which code executes with hardware privilege?
- Where do device drivers, filesystems and network stacks run?
- Does a service call use an in-kernel function call, protected IPC or a library call?
- Which component chooses resource-management policy?
- Can one faulty driver overwrite another subsystem?
- What interface must remain stable for applications?
- What can be verified or replaced independently?

#### **Monolithic, Microkernel, Exokernel, and Unikernel Designs**

A **monolithic kernel** places core services such as scheduling, virtual memory, VFS, networking and many drivers in one privileged address space. Internal calls and shared kernel data structures support high performance and flexible cross-subsystem optimization. Loadable modules add deployment modularity but usually run with the same privilege, so they do not create a hardware-enforced fault boundary.

A **microkernel** keeps a smaller set of mechanisms in privileged mode, commonly address spaces, threads/scheduling, IPC and capability or interrupt primitives. Filesystems, drivers and network services can run in separate user-mode processes. A driver fault can then be contained or restarted, but service requests cross protection domains and require carefully designed IPC, scheduling and memory-transfer paths.

An **exokernel** goes further in separating protection from management. The kernel securely multiplexes low-level hardware resources and exposes them to untrusted application-level library operating systems. A library OS implements abstractions such as virtual memory policy or filesystems chosen for an application. This enables specialization, but shifts complexity into libraries and requires low-level interfaces that preserve secure sharing, revocation and portability.

A **unikernel** links one application with selected OS library components into a specialized machine image, often using one address space and one privilege level inside a VM. Removing unneeded services can reduce image size and boot state, but traditional process isolation, dynamic administration and debugging assumptions may disappear. Isolation between unikernel instances normally comes from the VMM rather than an internal user/kernel boundary.

![Monolithic, microkernel, exokernel and unikernel designs place services, policy and protected boundaries differently.](assets/kernel-architecture-spectrum.svg){fig-alt="Four layered kernel-architecture columns compare monolithic in-kernel services, user-space microkernel servers, exokernel plus library OS, and a single-purpose unikernel image." width="98%"}

*Figure: original explanatory diagram based on the MIT [Exokernel paper](https://pdos.csail.mit.edu/6.828/2009/readings/engler95exokernel.pdf), the [seL4 microkernel architecture](https://sel4.systems/About/seL4-whitepaper.pdf), Linux kernel documentation, and the [Unikraft architecture](https://unikraft.org/docs/internals/architecture).*

| Architecture | Privileged core | Service path | Main benefit | Main engineering pressure |
|---|---|---|---|---|
| Monolithic | Most OS subsystems and drivers | In-kernel calls/shared state | Performance, compatibility, integrated optimization | Large privileged failure domain and complex evolution |
| Microkernel | Minimal scheduling, IPC, address-space and protection mechanisms | Protected IPC to user services | Fault isolation, small analyzable kernel | IPC/scheduling cost and distributed service design |
| Exokernel | Secure resource multiplexing | App-selected library OS uses low-level interface | Application-specific policy and flexibility | Portability, revocation, sharing and library complexity |
| Unikernel | App plus selected OS libraries in one image | Mostly direct library calls | Specialization, small boot surface | Tooling, updates, compatibility and intra-image isolation |

The boundaries can be hybrid. Linux is usually described as monolithic and modular, yet it can delegate filesystems through FUSE, networking through user-space frameworks, and selected policy through eBPF. A microkernel system may place several servers in one domain for performance. Unikraft deliberately combines modular library components with a single-address-space image. Architectural names identify a dominant organization, not a complete security proof.

### **Teaching Kernels and Production Kernels**

A kernel designed for teaching has a different optimization target from a kernel supporting billions of devices or a kernel serving as a formally verified security foundation. Judging all three by line count or feature count misses their purpose.

#### **xv6, Linux, and Verified Microkernels**

**xv6** is a small Unix-like teaching kernel used by MIT. Its value is end-to-end visibility: a student can follow a trap from assembly entry into C, trace process scheduling, inspect page-table creation, and understand pathname lookup and write-ahead logging without crossing millions of lines or many hardware backends. Its missing features are often deliberate simplifications, not defects in its teaching objective.

**Linux** is a production general-purpose kernel. It supports many architectures, filesystems, network protocols, security modules, virtual machines, containers and devices while maintaining long-lived user-space ABIs. Its design reflects performance, backward compatibility, hardware errata, concurrency, hotplug, observability and continuous change. Understanding one Linux path often requires tracing indirection, configuration-dependent implementations and subsystem contracts.

**seL4** is a capability-based microkernel accompanied by machine-checked proofs for specified configurations and properties. The functional-correctness result connects an abstract specification to the C implementation under stated assumptions; further proofs address properties such as integrity and confidentiality. The proof does not automatically verify arbitrary user services, device hardware, application policy or every deployment configuration.

![xv6, Linux and seL4 optimize respectively for readable mechanisms, broad production engineering and proof-backed kernel assurance.](assets/teaching-production-verified-kernels.svg){fig-alt="Three-column comparison of xv6 teaching simplicity, Linux production breadth, and seL4 capability microkernel verification with explicit proof scope." width="97%"}

*Figure: original explanatory diagram based on MIT's [xv6 course and book](https://pdos.csail.mit.edu/6.828/2021/xv6.html), the [Linux kernel documentation](https://docs.kernel.org/), and seL4's [proof assumptions](https://sel4.systems/Verification/assumptions.html).*

| Question | xv6 | Linux | seL4 |
|---|---|---|---|
| Primary goal | Teach classical OS mechanisms | General-purpose production platform | High-assurance minimal kernel foundation |
| Scope | Small Unix-like kernel and selected devices | Broad architectures, drivers and policies | Minimal capability microkernel plus separate user services |
| Assurance method | Readability, labs, reasoning and tests | Review, testing, sanitizers, fuzzing, static analysis and deployment experience | Formal specification and machine-checked proofs under assumptions |
| Best use while learning | Trace complete causal paths | Study real engineering, interfaces and scale | Learn precise specifications, capabilities and proof boundaries |
| Invalid conclusion | "All production kernels should be this small" | "Feature breadth proves correctness" | "The proof makes the whole deployed system secure" |

The three are complementary. A productive learning path uses xv6 to build a mechanism-level mental model, Linux to discover exceptions and engineering constraints, and seL4 to see how an informal security claim must be reduced to explicit state, invariants and assumptions.

### **Modern Kernel Mechanisms**

Modern kernels must scale across many cores, evolve without replacing established ABIs and expose observability without adding a custom syscall for every question. RCU and eBPF illustrate two different responses: one changes how shared state is updated; the other adds a controlled way to extend selected execution paths.

#### **RCU, eBPF, and Extensible Observability**

**Read-copy-update (RCU)** is a synchronization family optimized for read-mostly data. Instead of making every reader acquire a contended lock, an updater separates change into phases:

1. allocate and fully initialize a new object or new linkage;
2. atomically publish the new pointer or unlink the old object;
3. wait for an RCU **grace period**, long enough that readers which could have observed the old pointer have left their read-side critical sections; and
4. reclaim the old object only after that lifetime guarantee holds.

![RCU lets pre-existing readers finish on old state before reclaiming it after a grace period.](assets/rcu-publish-grace-reclaim-animated.svg){fig-alt="Animated RCU timeline showing publication of a new object, old and new readers, a grace period, and deferred reclamation of old state." width="96%"}

*Figure: original explanatory diagram based on the official Linux [RCU concepts](https://docs.kernel.org/RCU/rcu.html) and [RCU list guidance](https://docs.kernel.org/RCU/listRCU.html). It is distinct from the lock-oriented RCU timeline in the synchronization chapter and focuses on publication and reclamation.*

RCU's advantage comes from moving work away from readers. On supported read-side paths, readers avoid a conventional lock and often avoid writes to shared cache lines. The tradeoff is conceptual: update-side code must publish pointers with the correct ordering, preserve object lifetime, choose an appropriate RCU flavor and distinguish removal from reclamation.

RCU does **not** automatically make every compound invariant consistent. A reader may need a lock, sequence counter or version scheme when several fields must be observed as one atomic snapshot. Nor does `synchronize_rcu()` wait for every task in the system; it waits for the relevant pre-existing RCU read-side critical sections under that flavor's quiescent-state definition.

<details>
<summary><strong>RCU-protected replacement pattern in kernel-style C</strong></summary>

```c
// Simplified kernel-style pattern. Real code must choose the correct RCU
// flavor, allocation context, list primitives, locking, and error handling.

struct config {
    int threshold;
    struct rcu_head rcu;
};

static struct config __rcu *current_config;

int read_threshold(void)
{
    int value;

    // The read-side section guarantees that an object reached through the
    // RCU pointer will not be reclaimed until after rcu_read_unlock().
    rcu_read_lock();
    struct config *cfg = rcu_dereference(current_config);
    value = cfg->threshold;
    rcu_read_unlock();

    return value;
}

int replace_threshold(int new_value)
{
    struct config *new_cfg = kmalloc(sizeof(*new_cfg), GFP_KERNEL);
    if (!new_cfg)
        return -ENOMEM;

    // Finish initialization before publishing the pointer.
    new_cfg->threshold = new_value;

    // A separate update lock would be needed if multiple writers can race.
    struct config *old = rcu_replace_pointer(current_config, new_cfg, true);

    // Readers that began before replacement may still use old. Queue its
    // release after a grace period instead of freeing it immediately.
    if (old)
        kfree_rcu(old, rcu);

    return 0;
}
```

</details>

**eBPF** provides a restricted instruction set and kernel framework for loading programs that execute at defined hooks. Programs can observe tracepoints, function entry/exit, networking events, cgroup operations, perf events and selected security hooks. They exchange state with user space and each other through typed map abstractions, ring buffers and helper functions.

The critical path is not "upload arbitrary C into the kernel":

1. source is compiled to eBPF bytecode and metadata;
2. a privileged or otherwise authorized loader creates maps and requests program loading;
3. the kernel verifier explores control-flow states, tracks register and pointer types, checks initialization and bounds, and enforces program-type rules;
4. accepted bytecode can be interpreted or JIT-compiled to native instructions;
5. the program is attached to a compatible hook; and
6. events invoke it with a constrained context and permitted helpers.

![eBPF programs cross a verifier before JIT or interpretation and attachment to a typed kernel hook.](assets/ebpf-load-verify-attach.svg){fig-alt="eBPF architecture from source and loader through kernel verifier, reject path, JIT or interpreter, hook attachment, event execution, maps and user-space controller." width="98%"}

*Figure: original explanatory diagram based on Linux's [BPF documentation](https://docs.kernel.org/bpf/), the [eBPF verifier](https://docs.kernel.org/bpf/verifier.html), and the eBPF project's [loader and verification overview](https://ebpf.io/what-is-ebpf/).*

The verifier proves a defined class of safety properties under its model, such as controlled memory access and bounded execution paths. It does not decide whether a privileged network policy drops the right packets, whether a trace program leaks sensitive metadata, or whether an aggregation is statistically meaningful. Verifier and JIT defects are also kernel-security concerns because accepted programs execute in a privileged context.

<details>
<summary><strong>Observe the running pipeline with bpftrace</strong></summary>

```bash
# Run in one terminal with appropriate tracing privilege. Tracepoints are
# preferred over arbitrary function probes when a stable tracepoint exists.
sudo bpftrace -e '
tracepoint:syscalls:sys_enter_execve
/comm == "bash" || comm == "sh"/
{
  printf("exec caller pid=%d comm=%s file=%s\n", pid, comm,
         str(args->filename));
}

tracepoint:sched:sched_process_exec
{
  printf("exec complete pid=%d comm=%s file=%s\n", pid, comm,
         str(args->filename));
}

tracepoint:sched:sched_process_exit
/comm == "cat" || comm == "grep"/
{
  printf("exit pid=%d comm=%s\n", pid, comm);
}'

# In another terminal, trigger the case study.
cat input.txt | grep kernel > result.txt
```

</details>

This probe observes lifecycle events without modifying `cat`, `grep` or the shell. Production tracing must bound map growth and event volume, avoid expensive per-event formatting, pin to stable interfaces where possible, protect sensitive data and remove attachments predictably. Observability code is still code on a hot kernel path.

RCU and eBPF also interact. eBPF maps and attachment structures need scalable concurrency and safe object lifetime; kernel internals commonly use RCU to allow fast lookups while updates replace or detach state. One mechanism provides an update discipline, the other a verified extension framework.

### **Research and Evolution Frontiers**

The abstractions in this chapter continue to evolve because cloud workloads ask for combinations classical systems did not optimize together: mutually distrustful tenants, millisecond-scale startup, machine-level compatibility, operator-independent confidentiality, high accelerator throughput, verifiable isolation and application-specific OS services.

Research should be evaluated by identifying what it moves outside the TCB and what new assumptions replace it. A smaller VMM may depend more strongly on a host kernel. Encrypted guest memory may rely on a hardware security processor and attestation service. A library OS may remove a full guest kernel but implement a large compatibility surface beside the application.

#### **Confidential Computing, Library OSes, and Serverless Isolation**

**Confidential computing** aims to protect data while it is being processed. In a confidential VM, hardware such as AMD SEV-SNP, Intel TDX or Arm CCA can protect private guest memory and register state from direct host or VMM inspection. A security manager controls transitions and measurements, while **remote attestation** lets an external verifier assess evidence about the initial guest state before provisioning secrets.

A simplified attestation flow is:

1. the platform measures firmware, boot components or an initial guest image into a report;
2. hardware signs or authenticates evidence rooted in a platform identity;
3. the guest sends evidence plus a freshness challenge to a verifier;
4. the verifier checks signature chains, revocation, policy and measurement values; and
5. only then does a relying service release a key, credential or workload secret through a secure channel.

Attestation is evidence about an initial or measured state, not a prediction that code is vulnerability-free. It requires a policy that says which measurements are accepted and a lifecycle for firmware updates, revocation and key release. The untrusted host can still deny CPU, memory, network or storage service. Shared I/O buffers, device emulation, interrupts, time and side channels remain attack surfaces, so a confidential guest must validate host-provided input more defensively than a traditional guest.

**Library operating systems** move application-facing OS abstractions into a user-space library linked with or loaded beside an application. Gramine, for example, implements much of a Linux-facing environment above a narrow Platform Adaptation Layer (PAL). A backend can target a normal host or an enclave. The library OS handles some syscalls internally and funnels others through the PAL, validating responses where the host is outside the trust boundary.

This design can:

- avoid booting an entire general-purpose guest OS;
- specialize services for one application;
- narrow the interface exposed to an untrusted host;
- run existing binaries in environments such as enclaves; and
- make application and OS-library state one deployment unit.

The cost is compatibility engineering. Linux applications rely on subtle syscall, `/proc`, signal, thread, filesystem and timing behavior. Reimplementing the visible ABI correctly and securely can be substantial, and the host interface must resist malicious return values and Iago-style attacks.

**Serverless isolation** needs rapid creation and high density without placing mutually untrusted customer code directly in one kernel boundary. Firecracker addresses this with a specialized VMM built on KVM, a deliberately small virtual device model, a control API and an external jailer. A microVM still has a guest kernel, but the machine model is narrowed for function and container workloads rather than general desktop hardware.

Snapshots can further reduce cold start by restoring a prepared guest state. Snapshotting changes the security and correctness problem: random-generator state, network connections, clocks, machine identifiers, secrets and per-instance uniqueness may need regeneration or late binding. A fleet must also patch base snapshots and prove which version each restored instance uses.

![Confidential VMs, library OSes and serverless microVMs each move a different trust or compatibility boundary.](assets/modern-isolation-frontiers.svg){fig-alt="Three architecture stacks compare confidential virtual machines, library operating systems with a narrow PAL, and specialized serverless microVMs." width="98%"}

*Figure: original explanatory diagram based on Linux's [confidential-computing threat model](https://docs.kernel.org/security/snp-tdx-threat-model.html), the [Gramine architecture and PAL](https://gramine.readthedocs.io/en/stable/), and the USENIX [Firecracker paper](https://www.usenix.org/conference/nsdi20/presentation/agache).*

| Frontier | Boundary change | Intended benefit | Residual or new challenge |
|---|---|---|---|
| Confidential VM | Host/VMM excluded from private guest state | Protect data in use from a compromised operator layer | Availability, side channels, shared I/O, attestation policy |
| Library OS | App-facing OS services move into a library; host ABI narrows | Specialization and portability to enclaves or unusual hosts | Linux compatibility and hostile-host validation |
| Unikernel | App and selected OS components become one image | Small boot state and application-specific composition | Debugging, updates, ABI support and internal fault isolation |
| MicroVM | VM device model specialized for cloud workload | Hardware VM boundary with lower lifecycle overhead | Guest-kernel memory, snapshot hygiene and fleet orchestration |
| Sandboxed container | User-space kernel or syscall mediation added before host | Reduce direct host-kernel interface | Translation overhead and incomplete semantics |
| Verified microkernel | Small kernel contract tied to machine-checked proof | High-assurance protection foundation | User services, policy, devices and proof assumptions remain |

These designs increasingly combine. A container image can be unpacked into a microVM. A library OS can run inside a confidential VM or enclave. A verified microkernel can host a VMM or trusted service. The right question is not which label wins, but which composition yields a supportable interface and defensible TCB for the workload.

### **Running the Pipeline Natively, in a Container, and in a VM**

The same source-level pipeline reveals how boundary placement changes execution without changing shell syntax:

```bash
cat input.txt | grep kernel > result.txt
```

**Natively**, the shell asks the host kernel to create a pipe, fork or clone children, duplicate descriptors and execute `cat` and `grep`. The host VFS opens `input.txt` and `result.txt`; the host scheduler runs both processes; the host page cache and filesystem own buffered and durable state.

**In a shared-kernel container**, the syscall sequence is still handled by that host kernel. PID values are interpreted through a PID namespace, path lookup starts from the container mount namespace and root, credentials are interpreted with user-namespace mappings where configured, and resource use is charged to a cgroup. Image layers may supply `/bin/cat` and `/bin/grep`; bind mounts or volumes supply input and persistent output.

**In a VM**, the shell enters the guest kernel. The guest allocates its own pipe object, descriptors and process IDs. Guest VFS and page cache state eventually becomes virtual block or network I/O. A virtio driver publishes requests, a VMM or host backend services them, and the host scheduler decides when the vCPU and backend threads run. The guest sees completion as if it came from a device.

![The pipeline keeps its Unix process semantics while ownership of scheduling, path lookup, memory and I/O moves across native, container and VM stacks.](assets/pipeline-native-container-vm.svg){fig-alt="Three detailed stacks trace the cat grep shell pipeline through a host kernel, a container runtime boundary with shared kernel, and a guest kernel plus hypervisor." width="98%"}

*Figure: original explanatory diagram based on the Linux process, namespace, cgroup, KVM and virtio interfaces developed throughout this operating-systems series.*

The data path makes the distinction concrete:

| Event | Native | Container | VM |
|---|---|---|---|
| `pipe2()` | Host kernel creates pipe | Same host kernel; task belongs to container domains | Guest kernel creates guest pipe |
| `execve("cat")` | Host VFS resolves host path | Host VFS resolves container mount/root view | Guest VFS resolves guest path |
| Read input | Host page cache/filesystem | Host filesystem, image layer or mounted volume | Guest filesystem then virtual storage backend |
| Schedule `grep` | Host scheduler | Host scheduler with cgroup policy | Guest scheduler selects task; host scheduler selects vCPU |
| Write output | Host file and page cache | Container upper layer or mounted persistent storage | Guest page cache/filesystem then virtual device and host backing |
| Observe PID | Host namespace PID | Different inside and outside PID views | Guest PID has no direct host task equivalence beyond vCPU/VMM threads |
| Kernel fault | Can affect host | Can affect host and sibling containers | Guest kernel fault normally remains inside VM boundary |

<details>
<summary><strong>Run and compare the pipeline in three environments</strong></summary>

```bash
# Prepare one deterministic input in a new working directory.
demo_dir=$(mktemp -d)
cd "$demo_dir"
printf '%s\n' \
  'user code enters through a system call' \
  'the kernel creates and schedules both processes' \
  'a container shares the host kernel' \
  'a virtual machine has a guest kernel' > input.txt

# 1. Native: host shell, host processes, host kernel, host filesystem.
cat input.txt | grep kernel > result.native.txt

# Observe the process/file-descriptor syscalls without tracing every library call.
strace -f -o native.trace \
  -e trace=pipe,pipe2,clone,fork,vfork,execve,dup2,close,openat,read,write \
  sh -c 'cat input.txt | grep kernel > result.native.traced.txt'

# 2. Container: the commands come from the image, while this directory is
# mounted as data. The processes still enter the host Linux kernel.
podman run --rm \
  --network none --cap-drop all --security-opt no-new-privileges \
  --memory 128m --pids-limit 32 \
  -v "$PWD:/work:Z" -w /work \
  docker.io/library/alpine:latest \
  sh -c 'cat input.txt | grep kernel > result.container.txt'

# 3. VM: copy input into a prepared Linux VM and run through its guest kernel.
# Set VM_HOST to an SSH target you administer, such as student@192.0.2.10.
: "${VM_HOST:?set VM_HOST to your Linux VM SSH target}"
scp input.txt "$VM_HOST:/tmp/os11-input.txt"
ssh "$VM_HOST" \
  "cat /tmp/os11-input.txt | grep kernel > /tmp/os11-result.txt"
scp "$VM_HOST:/tmp/os11-result.txt" result.vm.txt

# Semantic output should match even though the execution paths differ.
sha256sum result.native.txt result.container.txt result.vm.txt

# Host-side observations expose different layers.
printf 'native/container host: '; uname -r
podman inspect --format \
  'container pid={{.State.Pid}} image={{.ImageName}}' \
  --latest 2>/dev/null || true
ssh "$VM_HOST" 'printf "guest kernel: "; uname -r; cat /proc/1/cgroup'
```

</details>

The hashes test output equivalence, not performance equivalence. A meaningful benchmark would warm or explicitly control caches, separate image transfer from execution, measure CPU and wall time, repeat enough samples, capture tail latency, record host load and inspect both guest and host counters. Tracing itself can perturb short workloads.

The experiment also highlights persistence. `result.native.txt` belongs directly to the host filesystem. `result.container.txt` survives because `/work` is an explicit bind mount; writing only to the container upper layer would make lifecycle semantics different. `result.vm.txt` first becomes guest filesystem state and is then copied through SSH; whether the original guest file is durable depends on guest and host storage stacks.

### **Comparison and Summary**

Virtualization is a continuation of the operating-system idea of controlled abstraction. A process receives a virtual address space; a container receives namespaced OS views and a resource domain; a VM receives a virtual machine. Each level multiplexes physical resources while attempting to preserve an interface and enforce a boundary.

The mechanisms can now be connected end to end:

| Mechanism | Abstraction presented | Fast path | Control or fault path | Central invariant |
|---|---|---|---|---|
| CPU virtualization | vCPU and guest privilege state | Direct guest instruction execution | VM exit, emulate/handle, VM entry | Guest cannot modify host control state directly |
| Memory virtualization | Guest-physical address space | TLB-cached composed mapping | Nested fault, mapping update, invalidation | GPA resolves only to host pages owned by that VM |
| I/O virtualization | Virtual device contract | Batched shared queue or assigned device | Notification, emulation, IOMMU fault, completion | Guest input is validated and DMA remains confined |
| Live migration | Movable running machine | Iterative page/device-state transfer | Pause, final state, ownership switchover | Exactly one authoritative coherent VM resumes |
| Container isolation | Namespaced, metered process environment | Ordinary host syscalls | Runtime setup and kernel policy checks | Workload sees and consumes only configured domains |
| Overlay filesystem | Merged image plus writable delta | Read shared lower object | Copy-up or whiteout | Immutable lower state is never silently mutated |
| RCU | Read-mostly shared object lifetime | Low-overhead read-side access | Publish, grace period, deferred reclaim | Old object outlives every reader that could see it |
| eBPF | Typed programmable kernel hook | Verified JIT/interpreted event execution | Load, verify, attach, reject or detach | Only accepted operations occur within hook constraints |

**The main architectural comparisons are:**

- Type 1 and Type 2 describe where VMM responsibilities sit, but hybrid designs must be decomposed by actual control path.
- Full virtualization preserves an unmodified guest contract; paravirtualization deliberately changes an internal interface to reduce expensive emulation.
- Shadow page tables compose mappings in software; nested paging moves the two-stage walk into hardware and makes caching decisive.
- Emulation maximizes compatibility, virtio improves mediated throughput, and passthrough trades management flexibility for a more direct device path.
- Containers share a host kernel and compose process-level controls; VMs place a virtual-hardware boundary below a guest kernel.
- Monolithic, microkernel, exokernel and unikernel designs differ primarily in where services and policy execute, how they communicate and what belongs to the protected core.
- xv6, Linux and seL4 should be read for different reasons: mechanism clarity, production engineering and proof-backed assurance.
- RCU and eBPF show how a mature kernel can scale reads and gain controlled extensibility without replacing its application ABI.

**A practical selection process is:**

1. define the adversary, assets and failure domains;
2. identify the lowest layer that must be outside the adversary's control;
3. list the complete TCB, including management and supply-chain components;
4. map required OS, device, snapshot, migration and observability contracts;
5. measure the workload's exits, TLB behavior, I/O queues, memory pressure and startup path;
6. choose container, sandbox, VM, microVM, library OS or a composition based on those facts;
7. design update, attestation, secret, backup and incident-response lifecycles; and
8. test boundary failure, not only normal throughput.

The strongest recurring lesson is that **a boundary is a claim about control, not a product name**. A container boundary depends on one host kernel and its configuration. A VM boundary depends on hypervisor, VMM, virtual devices, host components and hardware. A confidential VM changes which of those components may inspect private state but retains availability and interface risks. A verified kernel narrows uncertainty inside its proof scope but does not verify the surrounding system automatically.

For the running pipeline, every architecture must still preserve the Unix facts developed across this series: descriptors reference kernel objects, pipe EOF depends on closing all writers, the scheduler determines interleaving, address translation protects memory, VFS resolves names, buffered writes become durable only through a storage contract, and security depends on stable subjects, objects and policy. Virtualization changes **which kernel owns each fact and where the next boundary begins**.
